In [1]:
%pip install -r requirements.txt
import pandas as pd
import numpy as np

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
Note: you may need to restart the kernel to use updated packages.


In [2]:
pain_points = '../output/challenges.csv'
expectations = '../output/expectations.csv'

df_pain_points = pd.read_csv(pain_points)
df_expectations = pd.read_csv(expectations)

In [3]:
df_pain_points.shape, df_expectations.shape

((424, 2), (433, 2))

In [4]:
# remove records with less than or equal to 4 words
df_pain_points = df_pain_points[df_pain_points['pain_points'].str.split().str.len() > 4]
df_expectations = df_expectations[df_expectations['expectations'].str.split().str.len() > 4]

In [5]:
df_pain_points.shape, df_expectations.shape

((324, 2), (309, 2))

In [6]:
df_pain_points.sample(5)

,focus_group,pain_points
139,DOCE CommercialPermitElectrical Inspectors,Internet Dependency: Some functions are unavai...
55,CPO CPO Co-Ordinator,Address Lookup Challenges: Difficulty locating...
421,DOCE Housing Inspectors,Violation Entry Delays: Simultaneous data entr...
392,DOCE Housing Inspectors,No Global Case View: Difficult to review all a...
403,DOCE Housing Inspectors,Information Fragmentation: Important informati...


In [7]:
# Sentiment analysis for pain points (keyword-based, no VADER)
NEGATIVE_HINTS = [
    "challenge", "difficult", "difficulty", "problem", "issue", "lack", "limited",
    "cannot", "can't", "unable", "delay", "slow", "confusing", "complex",
    "fragmented", "scattered", "manual", "duplicate", "inconsistent", "error",
    "not", "missing", "barrier", "risk", "hard", "pain", "bottleneck",
    "too many", "overload", "volume", "split", "crash", "crashes", "failure", "outage",
    "expiration", "renewal", "backlog", "queue", "rework"
]

POSITIVE_HINTS = [
    "improve", "efficient", "streamline", "easy", "clear", "fast", "better",
    "good", "excellent", "success", "helpful", "benefit", "effective"
]

def classify_pain_point_sentiment(text: str):
    t = str(text).lower()
    neg_hits = sum(kw in t for kw in NEGATIVE_HINTS)
    pos_hits = sum(kw in t for kw in POSITIVE_HINTS)

    score = neg_hits - pos_hits

    if score >= 1:
        label = "negative"
    elif score <= -1:
        label = "positive"
    else:
        label = "neutral"

    return label

df_pain_points["sentiment"] = df_pain_points["pain_points"].apply(classify_pain_point_sentiment)
df_pain_points[["focus_group", "pain_points", "sentiment"]].sample(5, random_state=42)

,focus_group,pain_points,sentiment
171,NBD NBD Internal,Contact Management Issues: The IPS contact/add...,negative
143,DOCE CommercialPermitElectrical Inspectors,Lack of Coordination: Electrical inspectors id...,negative
176,NBD NBD Internal,Limited Data Sharing: Information is not easil...,negative
14,CPO Central Permit Office,Review Status Gaps: Difficult for applicants t...,negative
234,DOCE Admin Aide,Staff lose significant time handling repeat ca...,neutral


In [8]:
# Quick check for CPO rows mentioned
df_pain_points.loc[
    df_pain_points["focus_group"].eq("CPO Central Permit Office"),
    ["focus_group", "pain_points", "sentiment"]
] .head(10)

,focus_group,pain_points,sentiment
3,CPO Central Permit Office,Split Permit & Code Systems: Permit and Code E...,negative
4,CPO Central Permit Office,Scattered Information: Property and permit inf...,negative
5,CPO Central Permit Office,Research Complexity: Users must search multipl...,negative
6,CPO Central Permit Office,Difficult Information Retrieval: Finding relev...,negative
8,CPO Central Permit Office,Limited IPS-Camino Integration: Information do...,negative
9,CPO Central Permit Office,No Shared Permit/Violation Visibility: Permit ...,negative
10,CPO Central Permit Office,Duplicate Research Effort: Staff repeatedly se...,negative
11,CPO Central Permit Office,No Automatic Cross-System Notifications: Permi...,negative
13,CPO Central Permit Office,Limited Status Transparency: Applicants cannot...,negative
14,CPO Central Permit Office,Review Status Gaps: Difficult for applicants t...,negative


In [9]:
df_pain_points.loc[df_pain_points['sentiment'] == 'negative'].shape, df_pain_points.loc[df_pain_points['sentiment'] == 'positive'].shape, df_pain_points.loc[df_pain_points['sentiment'] == 'neutral'].shape

((235, 3), (2, 3), (87, 3))

In [10]:
# total pain points per focus group (show all groups)
pain_points_summary = df_pain_points.groupby("focus_group", dropna=False).agg(
    total_pain_points=pd.NamedAgg(column="pain_points", aggfunc="count"),
    negative_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "negative").sum()),
    positive_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "positive").sum()),
    neutral_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "neutral").sum())
).reset_index()

# Validation: these two numbers should match
print("distinct focus groups in data:", df_pain_points["focus_group"].nunique(dropna=False))
print("rows in summary:", len(pain_points_summary))

pain_points_summary.sort_values(by=["total_pain_points", "focus_group"], ascending=[False, True])

distinct focus groups in data: 16
rows in summary: 16


,focus_group,total_pain_points,negative_pain_points,positive_pain_points,neutral_pain_points
6,DOCE Admin Aide,34,17,0,17
8,DOCE CommercialPermitElectrical Inspectors,30,26,0,4
10,DOCE Housing Inspectors,30,22,1,7
7,DOCE Building Inspectors,29,20,0,9
11,DOCE Office Manager,28,17,1,10
12,DOCE Supervisors,26,19,0,7
9,DOCE Fire Prevention Bureau,25,18,0,7
4,CPO CPO Co-Ordinator,22,18,0,4
5,CPO Central Permit Office,21,15,0,6
15,NBD NBD Internal,14,13,0,1


In [11]:
# show only the pain points that are positive
top_focus_groups = pain_points_summary.sort_values(by="total_pain_points", ascending=False).head(10)["focus_group"].tolist()
df_pain_points.loc[
    df_pain_points["focus_group"].isin(top_focus_groups)
    & df_pain_points["sentiment"].eq("positive"),
    ["focus_group", "pain_points", "sentiment"]
]

,focus_group,pain_points,sentiment
304,DOCE Office Manager,Searching across properties and cases is ineff...,positive
422,DOCE Housing Inspectors,Large Dropdown Lists: Long dropdown menus make...,positive


In [12]:
#convert positive sentiment to neutral for pain points
df_pain_points.loc[df_pain_points['sentiment'] == 'positive', 'sentiment'] = 'neutral'

In [13]:
# Sentiment analysis for expectations (keyword-based, no VADER)
EXPECTATION_POSITIVE_HINTS = [
    "improve", "improved", "improvement", "streamline", "streamlined", "efficient",
    "efficiency", "faster", "reduce", "reduced", "automation", "automated",
    "clear", "visibility", "transparent", "single source", "integration", "integrated",
    "accurate", "consistency", "reliable", "easy", "easier", "better", "support", "success"
]

EXPECTATION_NEGATIVE_HINTS = [
    "lack", "limited", "cannot", "can't", "unable", "missing", "delay", "slow", "error",
    "difficult", "complex", "manual", "duplicate", "inconsistent", "fragmented", "risk", "barrier"
]

def classify_expectation_sentiment(text: str):
    t = str(text).lower()
    pos_hits = sum(kw in t for kw in EXPECTATION_POSITIVE_HINTS)
    neg_hits = sum(kw in t for kw in EXPECTATION_NEGATIVE_HINTS)

    score = pos_hits - neg_hits

    if score >= 1:
        label = "positive"
    elif score <= -1:
        label = "negative"
    else:
        label = "neutral"

    return label

df_expectations["sentiment"] = df_expectations["expectations"].apply(classify_expectation_sentiment)

# total expectations per focus group (show all groups)
expectations_summary = df_expectations.groupby("focus_group", dropna=False).agg(
    total_expectations=pd.NamedAgg(column="expectations", aggfunc="count"),
    negative_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "negative").sum()),
    positive_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "positive").sum()),
    neutral_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "neutral").sum())
).reset_index()

# Validation: these two numbers should match
print("distinct focus groups in data:", df_expectations["focus_group"].nunique(dropna=False))
print("rows in summary:", len(expectations_summary))

expectations_summary.sort_values(by=["total_expectations", "focus_group"], ascending=[False, True])

distinct focus groups in data: 16
rows in summary: 16


,focus_group,total_expectations,negative_expectations,positive_expectations,neutral_expectations
10,DOCE Housing Inspectors,31,2,15,14
12,DOCE Supervisors,30,1,14,15
7,DOCE Building Inspectors,27,1,17,9
5,CPO Central Permit Office,26,0,19,7
6,DOCE Admin Aide,26,1,17,8
8,DOCE CommercialPermitElectrical Inspectors,26,1,16,9
9,DOCE Fire Prevention Bureau,26,0,18,8
4,CPO CPO Co-Ordinator,25,0,17,8
11,DOCE Office Manager,21,1,15,5
3,CPC CPC,13,0,11,2


In [14]:
# display expectations that are negative
neg_expectations = df_expectations.loc[
    df_expectations["sentiment"] == "negative",
    ["focus_group", "expectations", "sentiment"]
]
neg_expectations.sample(n=min(10, len(neg_expectations)), random_state=42)

,focus_group,expectations,sentiment
88,DOCE Building Inspectors,Consolidated Ownership Records: Eliminate dupl...,negative
141,DOCE CommercialPermitElectrical Inspectors,Unified Scheduling Platform: Eliminate duplica...,negative
397,DOCE Housing Inspectors,Missing Inspection Tracking: Report on cases w...,negative
215,DOCE Admin Aide,Eliminate duplicate data entry where possible.,negative
313,DOCE Office Manager,Elimination of duplicate data entry.,negative
234,DOCE Supervisors,Missed Case Identification: Automatically iden...,negative
427,DOCE Housing Inspectors,Lead Risk Identification: Automatically identi...,negative


In [15]:
#convert negative sentiment to neutral for expectations
df_expectations.loc[df_expectations['sentiment'] == 'negative', 'sentiment'] = 'neutral'

In [16]:
#save as csv
df_pain_points.to_csv("../output/clean_challenges.csv", index=False)
df_expectations.to_csv("../output/clean_expectations.csv", index=False)